Pre-process ERA5 data to reduce size. Crop 2t and 2d to Caribbean and save in single nc to work folder.

In [1]:
import xarray as xr
import glob
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import dask

# file check

In [2]:
ds = xr.open_dataset('/glade/campaign/cgd/cas/observations/ERA5/mon/t2m/era5.t2m.194001-202512.nc')
ds

<xarray.Dataset> Size: 4GB
Dimensions:     (valid_time: 1032, latitude: 721, longitude: 1440)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 8kB 1940-01-01 ... 2025-12-01
    expver      (valid_time) <U4 17kB ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    number      int64 8B ...
Data variables:
    t2m         (valid_time, latitude, longitude) float32 4GB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-09T06:48 GRIB to CDM+CF via cfgrib-0.9.1...

In [3]:
ds1 = xr.open_dataset('/gdex/data/d633000/e5.oper.an.sfc/194001/e5.oper.an.sfc.128_167_2t.ll025sc.1940010100_1940013123.nc')
ds1

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 744, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 6kB 1940-01-01 ... 1940-01-31T23:00:00
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    VAR_2T     (time, latitude, longitude) float32 3GB ...
    utc_date   (time) int32 3kB ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r1i7n2 4.12.14-95.51-default #1 SMP Fri Apr 1...
    CONVERSION_DATE:      Thu Mar 16 18:41:53 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Thu Mar 16 18:42:14 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

In [4]:
ds = xr.open_dataset('/gdex/data/d633000/e5.oper.an.sfc/194108/e5.oper.an.sfc.128_168_2d.ll025sc.1941080100_1941083123.nc')
ds

<xarray.Dataset> Size: 3GB
Dimensions:    (time: 744, latitude: 721, longitude: 1440)
Coordinates:
  * time       (time) datetime64[ns] 6kB 1941-08-01 ... 1941-08-31T23:00:00
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
Data variables:
    VAR_2D     (time, latitude, longitude) float32 3GB ...
    utc_date   (time) int32 3kB ...
Attributes:
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB1 data to netCDF4.
    NETCDF_VERSION:       4.8.1
    CONVERSION_PLATFORM:  Linux r13i7n26 4.12.14-95.51-default #1 SMP Fri Apr...
    CONVERSION_DATE:      Tue May  9 16:37:31 MDT 2023
    Conventions:          CF-1.6
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    history:              Tue May  9 16:37:48 2023: ncks -4 --ppc default=7 e...
    NCO:                  netCDF Operators version 5.0.3 (Homepage = http://n...

# file selection

In [2]:
t_files = sorted(glob.glob('/gdex/data/d633000/e5.oper.an.sfc/*/*2t*.nc'))
# t_files

In [3]:
d_files = sorted(glob.glob('/gdex/data/d633000/e5.oper.an.sfc/*/*2d*.nc'))
# d_files

# dask setup

In [8]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=25)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.178:36305,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


2026-05-30 10:03:38,522 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='jupyterhub.hpc.ucar.edu', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tornado/web.py", line 3409, in wrapper
    return method(self, *args, **kwargs)
  File "/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/bokeh/server/views/ws.py", line 149, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the 

# get data and crop

## hmm1

In [7]:
# # I have not found a way to add a variable to preprocess function,
# # doc mentions it takes a ds type but no info on adding other arguments,
# # end up with a function per veriable.
# def crop_ds(ds, var):
#     ds = ds[var]
#     ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180  # easier to use east west when slicing,
#     ds = ds.sortby(ds.longitude)
#     ds = ds.sortby(ds.latitude)
#     ds_crop = ds.sel(latitude=slice(10, 25), longitude=slice(-85, -60))
#     return ds_crop

In [22]:
# test = crop_ds(ds1, 'VAR_2T')
# test

In [23]:
# # open_mfdataset does not do chunks larger then filesize
# # zarr can be written by various workers but needs same size chunks which months are not
# # anything less than year chunk of hourly data are too large in number making process too slow when loading to analyze
# # to_netcdf cannot write to a single file with various workers
# # @dask.delayed(nout=len(t_files))
# def open_era(file_list, var):
#     out_ds = []
#     for file in file_list:
#         ds = xr.open_dataset(file)
#         ds = crop_ds(ds, var)
#         out_ds.append(ds)
#     xr.concat(out_ds, dim='time')
#     return out_ds

In [ ]:
# T = open_era(t_files, 'VAR_2T')
# D = open_era(d_files, 'VAR_2D')

## hmm2

In [9]:
def crop_t_ds(ds):
    ds = ds['VAR_2T']
    ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
    ds = ds.sortby(ds.longitude)
    ds = ds.sortby(ds.latitude)
    ds_crop = ds.sel(latitude=slice(10, 25), longitude=slice(-85, -60))
    return ds_crop

In [10]:
T = xr.open_mfdataset(t_files, combine='nested', concat_dim='time',
                     engine='h5netcdf', parallel=True,
                     preprocess=crop_t_ds,
                     chunks={})

In [13]:
# need a good factor of chunks
T.sizes

Frozen({'time': 755304, 'latitude': 61, 'longitude': 101})

In [19]:
T = T.chunk({'time': (T.sizes['time'] / 33), 'latitude': -1, 'longitude': -1})
T

<xarray.DataArray 'VAR_2T' (time: 755304, latitude: 61, longitude: 101)> Size: 19GB
dask.array<rechunk-merge, shape=(755304, 61, 101), dtype=float32, chunksize=(22888, 61, 101), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 488B 10.0 10.25 10.5 10.75 ... 24.5 24.75 25.0
  * longitude  (longitude) float64 808B -85.0 -84.75 -84.5 ... -60.25 -60.0
Attributes: (12/14)
    long_name:                                          2 metre temperature
    short_name:                                         2t
    units:                                              K
    original_format:                                    WMO GRIB 1 with ECMWF...
    ecmwf_local_table:                                  128
    ecmwf_parameter:                                    167
    ...                                                 ...
    grid_specification:                                 0.25 degree x 0.25 de...
    rda_dataset:                                        ds633.0
    rda_dataset_url:                                    https:/rda.ucar.edu/d...
    rda_dataset_doi:                                    DOI: 10.5065/BH6N-5N20
    rda_dataset_group:                                  ERA5 atmospheric surf...
    QuantizeGranularBitGroomNumberOfSignificantDigits:  7

In [20]:
def crop_d_ds(ds):
    ds = ds['VAR_2D']
    ds.coords['longitude'] = (ds.coords['longitude'] + 180) % 360 - 180
    ds = ds.sortby(ds.longitude)
    ds = ds.sortby(ds.latitude)
    ds_crop = ds.sel(latitude=slice(10, 25), longitude=slice(-85, -60))
    return ds_crop

In [21]:
D = xr.open_mfdataset(d_files, combine='nested', concat_dim='time',
                     engine='h5netcdf', parallel=True,
                     preprocess=crop_d_ds,
                     chunks={})
D = D.chunk({'time': (D.sizes['time'] / 33), 'latitude': -1, 'longitude': -1})

# output to reusable file

In [24]:
T.to_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/sfc_hourly_temp', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 580.73 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [ ]:
D.to_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/sfc_hourly_dew', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/distributed/client.py:3398: UserWarning: Sending large graph of size 580.56 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


# shutdown dask

In [26]:
client.shutdown()

In [19]:
quit()